# Melampo: misura dell'interprete delle immagini (VoxTell + TotalSegmentator)

Solo dati pubblici: Medical Segmentation Decathlon, Task06 Lung (tumori polmonari in TC, licenza CC BY-SA 4.0).

Prima di eseguire, nelle impostazioni del notebook (pannello a destra):
- **Accelerator**: GPU T4 x2 (oppure P100);
- **Internet**: on (serve la verifica del telefono sull'account Kaggle);
- se il repository è privato: **Add-ons → Secrets → GITHUB_TOKEN** con un token GitHub *fine-grained*, solo lettura, solo su `Melampo-AI`.

Poi *Run all*. Alla fine si scarica `/kaggle/working/probe.json` e il log, da portare a Claude.

In [ ]:
!pip install -q voxtell TotalSegmentator nibabel
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os, subprocess
token = ""
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    pass  # repository pubblico: nessun token
url = f"https://{token}@github.com/francescoluciolattari/Melampo-AI.git" if token else "https://github.com/francescoluciolattari/Melampo-AI.git"
subprocess.run(["git", "clone", "--depth", "1", url, "/kaggle/working/melampo"], check=True)
print(open("/kaggle/working/melampo/scripts/imaging_probe.py").read()[:300])

In [ ]:
# Medical Segmentation Decathlon, Task06 Lung (circa 8,5 GB), dal bucket usato da MONAI
!wget -q -O /kaggle/temp/Task06_Lung.tar https://msd-for-monai.s3-us-west-2.amazonaws.com/Task06_Lung.tar || echo DOWNLOAD_FAILED
!tar -xf /kaggle/temp/Task06_Lung.tar -C /kaggle/temp && ls /kaggle/temp/Task06_Lung | head

In [ ]:
import glob, json, os
N = 5   # casi da misurare; aumentare dopo il primo giro
images = sorted(glob.glob("/kaggle/temp/Task06_Lung/imagesTr/lung_*.nii.gz"))[:N]
lobes = ["lung_upper_lobe_left", "lung_lower_lobe_left", "lung_upper_lobe_right", "lung_middle_lobe_right", "lung_lower_lobe_right"]
cases = [{"id": os.path.basename(p).split(".")[0], "image": p, "lesion": p.replace("imagesTr", "labelsTr"),
          "structure": lobes, "prompt": "tumore polmonare", "prompt_en": "lung tumor"} for p in images]
json.dump(cases, open("/kaggle/working/cases.json", "w"), indent=1)
print(len(cases), "casi")

In [ ]:
!cd /kaggle/working/melampo && python scripts/imaging_probe.py --cases /kaggle/working/cases.json --out /kaggle/working/probe.json --device cuda --fast-organs 2>&1 | tee /kaggle/working/probe.log

In [ ]:
import json
rows = json.load(open("/kaggle/working/probe.json"))
for r in rows:
    for k in ("prompt", "prompt_en"):
        if k in r:
            v = r[k]
            print(f"{r['id']:10} {k:9} {v['status']:10} Dice {v['dice']:.2f}  {v['seconds']:6.1f}s  "
                  f"misurato {v['measured_longest_axial_mm']} mm  riferimento {v['reference_longest_axial_mm']} mm")